# 🎥 Video Query Pipeline Example

This notebook demonstrates how to use the unified **VideoQueryPipeline** to answer natural language questions about your video content. 

The pipeline supports two major modes of operation:
1.  **State Machine (Deterministic)**: High precision, predictable state transitions. Best for chapter-based or event-based queries where accuracy is paramount.
2.  **Graph Swarm (Agentic)**: High recall, agentic discovery. Best for broad cross-video searches, tracking subjects over long durations, or complex multi-step reasoning.

---

### 🛠️ Setup
Ensure your `.env` file is configured with the necessary service credentials (LLM, Embedding, Neo4j, etc.).

In [ ]:
import os
import nest_asyncio
from dotenv import load_dotenv
from mmct.video_pipeline.query_pipeline import VideoQueryPipeline, QueryPipelineMode

# Apply nest_asyncio for Jupyter compatibility
nest_asyncio.apply()

# Load environment variables from .env
load_dotenv()

### 🧠 Mode 1: State-Machine Query
The State Machine mode flows through deterministic stages (`PLAN` -> `RETRIEVE` -> `SYNTHESIZE`).

In [ ]:
# Initialize the pipeline in STATE mode
# Setting use_provider_defaults=True automatically hydrates all providers from your environment configs
state_pipeline = VideoQueryPipeline(
    mode=QueryPipelineMode.STATE,
    use_provider_defaults=True,
    use_critic=True
)

query = "user-query."
video_id = "video-id" # Replace with an actual ID from your Neo4j database

print(f"🚀 Running State Machine Query: {query}")
result = await state_pipeline.query(user_query=query, video_id=video_id)

print(f"\n✅ Answer: {result['answer']}")
print(f"\n📚 Sources found: {len(result['sources'])}")

### 🕸️ Mode 2: Graph-Swarm Query
The Graph mode uses an AutoGen Swarm to "search and traverse" the Neo4j graph agentically.

In [ ]:
# Initialize the pipeline in GRAPH mode
graph_pipeline = VideoQueryPipeline(
    mode=QueryPipelineMode.GRAPH,
    use_provider_defaults=True
)

query = "user-query" # cross video query initiation

print(f"🚀 Running Graph Swarm Query: {query}")
result = await graph_pipeline.query(user_query=query)

print(f"\n✅ Answer: {result['answer']}")
print(f"\n📊 Token Usage: {result['token_usage']}")

#### 🎯 Scoped Search (Single Video)
While Graph mode defaults to cross-video search, you can explicitly scope it to a single video or a specific list of videos by passing `video_id` or `video_ids`.

In [ ]:
# Initialize the pipeline in GRAPH mode
graph_pipeline = VideoQueryPipeline(
    mode=QueryPipelineMode.GRAPH,
    use_provider_defaults=True
)

query = "user-query"
video_id = "video-id" 

print(f"🚀 Running Graph Swarm Query: {query}")
result = await graph_pipeline.query(user_query=query,video_id=video_id)

print(f"\n✅ Answer: {result['answer']}")
print(f"\n📊 Token Usage: {result['token_usage']}")

### 🌊 Streaming Pipeline Events
You can also use `query_stream` to observe the pipeline as it transitions through different states.

In [ ]:
print("🌊 Starting Streamed Query...\n")

async for event in state_pipeline.query_stream("user-query",video_id="video-id"):
    if event["type"] == "message":
        print(f"[{event['agent']}] {event['content']}")
    elif event["type"] == "final":
        print(f"\n🏁 Final Result: {event['data']['answer']}")

### 🧹 Cleanup
Always close the pipeline to release Neo4j driver connections.

In [ ]:
await state_pipeline.close()
await graph_pipeline.close()
print("✨ Pipelines closed.")